# CADS lesion benchmark (Colab GPU)
3D Faster R-CNN vs batched per-slice classifier — memory + time.

Pipeline: **train both tracks → test both on held-out scans (quality) → benchmark both on those scans (latency + memory)**.
Small real subset, minimal training. Run top-to-bottom on a T4/L4.

Tip: upload the `modiface` folder to Google Drive once (Option B below) — code and `checkpoints/` then persist across sessions. Bulk NIfTI data goes to local Colab SSD (`DATA_ROOT`, section 0): Drive's FUSE mount drops under heavy I/O, while local disk is fast and reliable (re-download each session; it's quick).

In [ ]:
!nvidia-smi
!pip -q install torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip -q install nibabel huggingface_hub datasets pyyaml tqdm scikit-learn psutil matplotlib

## 0. Project code & paths — pick ONE option
Run **either** Option A (GitHub) **or** Option B (Google Drive), then the path check.

In [ ]:
# ── Option A: clone from GitHub ──
!git clone https://github.com/adoskk/MachineLearningBasics.git
%cd MachineLearningBasics/CV/CNN_vs_3DRCNN_latency

In [ ]:
# ── Option B: use the project folder you uploaded to Google Drive ──
# 1. In Drive, upload the CNN_vs_3DRCNN_latency folder (code only, no data/),
#    e.g. MyDrive/Colab Notebooks/CNN_vs_3DRCNN_latency
#    (bulk data goes to local SSD in section 0)
# 2. Set DRIVE_PROJECT below to that location, then run this cell
DRIVE_PROJECT = 'MyDrive/Colab Notebooks/CNN_vs_3DRCNN_latency'  # <-- edit if elsewhere

import os
from google.colab import drive
drive.mount('/content/drive')
PROJECT_ROOT = f'/content/drive/{DRIVE_PROJECT}'
os.chdir(PROJECT_ROOT)
print('cwd:', os.getcwd())

In [ ]:
# ── Path check: CWD == project root; bulk DATA on local SSD ──
# Drive FUSE disconnects under heavy I/O ('Transport endpoint is not
# connected'), so NIfTI data lives on ephemeral /content (fast, reliable)
# while code + checkpoints stay on Drive (persistent).
import os
assert os.path.exists('configs/config.yaml'), 'not in project root — run Option A or B first'
DATA_ROOT = '/content/data/cads'
os.makedirs(DATA_ROOT, exist_ok=True)
for d in ['checkpoints', 'results']:
    os.makedirs(d, exist_ok=True)
print('project root OK:', os.getcwd())
print('DATA_ROOT (local SSD):', DATA_ROOT)
!ls

## 1. Data — local SSD, re-download each session
Accept the terms at https://huggingface.co/datasets/mrmrx/CADS-dataset. Download targets `DATA_ROOT` (/content, fast local disk) and is skipped if volumes are already there. `--max-files` caps total NIfTI files (whole volumes: 1 image + one file per `--parts`); e.g. `120` = ~60 part-551 volumes, plenty for the default `n_volumes: 30`.

Drive disconnected? (`Transport endpoint is not connected`) — remount and resume from Option B:
`drive.mount('/content/drive', force_remount=True)`

In [ ]:
import subprocess
from pathlib import Path

cached = list(Path(DATA_ROOT).rglob('images/*.nii*'))
print('cached volumes:', len(cached))
if cached:
    print(f'using cached data in {DATA_ROOT} — rm -rf it to re-download')
else:
    from huggingface_hub import login
    login()  # paste token
    subprocess.run(['python', 'scripts/download_subset.py',
                      '--subsets', '0003_kits21', '0004_lits',
                      '--local-dir', DATA_ROOT,
                      '--max-files', '5000',
                      '--parts', '551'], check=True)

## 2. Train both tracks
One script per model — revisit either one independently (`src/detector/`, `src/classifier/`). Checkpoints land in `checkpoints/`.

In [ ]:
%%time
!python scripts/train_detector.py --config configs/config.yaml --data-root {DATA_ROOT}

In [ ]:
%%time
!python scripts/train_classifier.py --config configs/config.yaml --data-root {DATA_ROOT}

## 3. Test both on the held-out scans (quality)
Detector: AP@IoU3D. Classifier: slice AP / AUROC. Untimed — pure quality context for the benchmark.

In [ ]:
!python scripts/test_detector.py --config configs/config.yaml --data-root {DATA_ROOT}
!python scripts/test_classifier.py --config configs/config.yaml --data-root {DATA_ROOT}

## 4. Benchmark both on the test scans (latency + memory)
`--no-train` loads the trained checkpoints and times them on the same held-out split: warmup + synced repeats, peak CUDA MB, CPU RSS delta, plus the quality numbers above as context.

In [ ]:
!python scripts/run_benchmark.py --config configs/config.yaml --no-train --repeats 20 --data-root {DATA_ROOT}

import json
r = json.load(open('results/benchmark.json'))
print('detector:', round(r['detector']['mean_ms_per_volume'],1), 'ms/vol |',
      round(r['detector']['peak_cuda_mb'],1), 'peak CUDA MB | AP', round(r['detector']['ap_iou03'],3))
for k,v in r['slice_classifier'].items():
    if k.startswith('batch_'):
        print(k, round(v['mean_ms_per_volume'],1), 'ms/vol |', round(v['slices_per_sec'],0), 'slices/s |',
              round(v['peak_cuda_mb'],1), 'MB | AUROC', round(v['auroc'],3))

In [ ]:
# Plot: latency vs batch size, memory footprint, quality context
import matplotlib.pyplot as plt
bs, ms, mem, auroc = [], [], [], []
for k,v in r['slice_classifier'].items():
    if k.startswith('batch_'):
        bs.append(int(k.split('_')[1])); ms.append(v['mean_ms_per_volume'])
        mem.append(v['peak_cuda_mb']); auroc.append(v['auroc'])
fig, ax = plt.subplots(1,3, figsize=(14,3.5))
ax[0].plot(bs, ms, marker='o'); ax[0].axhline(r['detector']['mean_ms_per_volume'], ls='--', label='3D Faster R-CNN')
ax[0].set(xlabel='classifier batch size', ylabel='ms / volume', title='Latency: batching closes the gap'); ax[0].legend()
ax[1].bar(['detector']+[f'cls@{b}' for b in bs], [r['detector']['peak_cuda_mb']]+mem)
ax[1].set(ylabel='peak CUDA MB', title='Memory footprint')
ax[2].bar(['detector-AP']+[f'cls@{b}' for b in bs], [r['detector']['ap_iou03']]+auroc)
ax[2].set(ylabel='AP / AUROC', title='Quality context (not same metric!)'); plt.tight_layout(); plt.show()